# Data Collection: ChatGPT (OpenAI API)

This notebook queries GPT-4o via the OpenAI API for each question in the question bank and saves responses to `data/responses/openai.jsonl`.

**Before running:** complete the setup in [Environment Setup](setup.md) and ensure `OPENAI_API_KEY` is set.

In [ ]:
import openai
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

In [ ]:
MODEL = "gpt-4o"                    # or gpt-4-turbo
TEMPERATURE = 0.0
MAX_TOKENS = 1024
OUTPUT_FILE = Path("data/responses/openai.jsonl")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about early church history.
Provide accurate, detailed, and nuanced responses based on historical scholarship.
Acknowledge uncertainty or scholarly debate where it exists.
Do not refuse historically answerable questions on the grounds of contemporary theological controversy."""

questions = pd.read_csv("../figures/question_bank.csv")
print(f"Loaded {len(questions)} questions")

In [ ]:
def query_openai(question_text: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question_text},
        ],
    )
    return {
        "response": response.choices[0].message.content,
        "tokens_used": response.usage.total_tokens,
        "model_version": response.model,
    }


with open(OUTPUT_FILE, "w") as f:
    for _, row in questions.iterrows():
        print(f"Querying {row['question_id']}...", end=" ")
        try:
            result = query_openai(row["question"])
            record = {
                "question_id": row["question_id"],
                "figure": row["figure"],
                "model": MODEL,
                "model_version": result["model_version"],
                "prompt": row["question"],
                "response": result["response"],
                "temperature": TEMPERATURE,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "tokens_used": result["tokens_used"],
            }
            f.write(json.dumps(record) + "\n")
            print(f"done ({result['tokens_used']} tokens)")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.5)

print(f"\nDone. Responses saved to {OUTPUT_FILE}")